### Example Fixed Project

The rest of this tutorial uses pre compiled ORBIT configs that are stored as .yaml files in the '~/configs/ folder. There are load and save methods available in ORBIT for working with .yaml files. These example projects each exhibit different functionalities within ORBIT. Using these examples and combinations of them, most project configurations can be modeled. 

In [1]:
import os
import pandas as pd
from ORBIT import ProjectManager, load_config
import csv

weather = pd.read_csv("data/example_weather.csv", parse_dates=["datetime"])\
            .set_index("datetime")

Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.

### Load the project configuration

In [2]:
fixed_config = load_config("configs/COE_2022_fixed_bottom_15_MW.yaml")  # Configs can be loaded with absolute or relative paths

print(type(fixed_config))                                         # They are loaded in as dictionaries.

print(f"Num turbines: {fixed_config['plant']['num_turbines']}")   # Once a configuration is loaded, different parameters can  
print(f"Turbine: {fixed_config['turbine']}")                      # be accessed using dict access.
print(f"\nSite: {fixed_config['site']}")

<class 'dict'>
Num turbines: 40
Turbine: 15MW_generic

Site: {'depth': 34, 'distance': 116, 'distance_to_landfall': 50, 'mean_windspeed': 9.17}


### Phases

This fixed project represents a generic Offshore Wind farm with 50 6MW turbines. It includes 5 design modules and 6 installation modules as seen below. This is a common set of modules to run for a fixed bottom project. This config will model the procurement and installation of monopiles, scour protection, array system, export system, offshore substation and the turbines.

In [3]:
print(f"Design phases: {fixed_config['design_phases']}")
print(f"\nInstall phases: {list(fixed_config['install_phases'].keys())}")

Design phases: ['ArraySystemDesign', 'ExportSystemDesign', 'MonopileDesign', 'ScourProtectionDesign', 'OffshoreSubstationDesign']

Install phases: ['ArrayCableInstallation', 'ExportCableInstallation', 'MonopileInstallation', 'OffshoreSubstationInstallation', 'ScourProtectionInstallation', 'TurbineInstallation']


### Run

This project is always being modeled with the example weather project supplied that is representative of US East Coast wind farm locations.

In [4]:
project = ProjectManager(fixed_config, weather=None)
project.run()

ORBIT library intialized at 'C:\upsizing_ORBIT\ORBIT\library'


### Top Level Outputs

ProjectManager offers several high level result categories:
- Installation CapEx
- System CapEx (procurement of BOS subcomponents)
- Turbine CapEx
- Soft CapEx (project management costs)
- Total CapEx
- Total installation time
- etc.

In [5]:
print(f"Installation CapEx:  {project.installation_capex/1e6:.0f} M")
print(f"System CapEx:        {project.system_capex/1e6:.0f} M")
print(f"Turbine CapEx:       {project.turbine_capex/1e6:.0f} M")
print(f"Soft CapEx:          {project.soft_capex/1e6:.0f} M")
print(f"Total CapEx:        {project.total_capex/1e6:.0f} M")

print(f"\nInstallation Time: {project.installation_time:.0f} h")

Installation CapEx:  182 M
System CapEx:        534 M
Turbine CapEx:       1020 M
Soft CapEx:          326 M
Total CapEx:        2213 M

Installation Time: 9896 h


### CapEx Breakdown

In [6]:
# The breakdown of project costs by module is available  at 'capex_breakdown'
project.capex_breakdown

{'Array System': 35028632.734,
 'Export System': 100357800.0,
 'Offshore Substation': 83413425.0,
 'Scour Protection': 10003200,
 'Substructure': 305291988.55575705,
 'Array System Installation': 11078820.142834675,
 'Export System Installation': 9440257.203684013,
 'Offshore Substation Installation': 3361105.2115677316,
 'Scour Protection Installation': 16485022.83105022,
 'Substructure Installation': 61063129.74851519,
 'Turbine Installation': 80203592.08523594,
 'Turbine': 1020000000,
 'Soft': 325896000.0,
 'Project': 151250000.0}

### Installation Actions

In [7]:
df = pd.DataFrame(project.actions)    # The project simulation logs are also available for all modules
df

,cost_multiplier,agent,action,duration,cost,level,time,phase,location,phase_name,site_depth,max_waveheight,max_windspeed,transit_speed,hub_height,per_trip
0,0.5,Array Cable Installation Vessel,Mobilize,72.000000,1.800000e+05,ACTION,0.000000,ArrayCableInstallation,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0.5,Export Cable Installation Vessel,Mobilize,72.000000,1.800000e+05,ACTION,0.000000,ExportCableInstallation,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0.5,Export Cable Burial Vessel,Mobilize,72.000000,1.800000e+05,ACTION,0.000000,ExportCableInstallation,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,Onshore Construction,Onshore Construction,0.000000,4.446058e+06,ACTION,0.000000,ExportCableInstallation,Landfall,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,0.5,Heavy Lift Vessel,Mobilize,72.000000,7.500000e+05,ACTION,0.000000,OffshoreSubstationInstallation,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2472,NaN,WTIV,Attach Blade,3.500000,7.291667e+04,ACTION,4544.996989,TurbineInstallation,NaN,TurbineInstallation,34.0,NaN,NaN,NaN,150.0,NaN
2473,NaN,WTIV,Release Blade,1.000000,2.083333e+04,ACTION,4545.996989,TurbineInstallation,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2474,NaN,WTIV,Lift Blade,1.500000,3.125000e+04,ACTION,4547.496989,TurbineInstallation,NaN,TurbineInstallation,34.0,NaN,NaN,NaN,150.0,NaN
2475,NaN,WTIV,Attach Blade,3.500000,7.291667e+04,ACTION,4550.996989,TurbineInstallation,NaN,TurbineInstallation,34.0,NaN,NaN,NaN,150.0,NaN


In [8]:
# These logs can be sorted by phase by using DataFrame operations

turbine_install = df.loc[df['phase']=="TurbineInstallation"]
turbine_install

,cost_multiplier,agent,action,duration,cost,level,time,phase,location,phase_name,site_depth,max_waveheight,max_windspeed,transit_speed,hub_height,per_trip
578,1.0,WTIV,Mobilize,168.000000,3.500000e+06,ACTION,1297.523656,TurbineInstallation,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
582,NaN,WTIV,Fasten Tower Section,4.000000,8.333333e+04,ACTION,1301.523656,TurbineInstallation,NaN,TurbineInstallation,34.0,NaN,NaN,NaN,150.0,NaN
583,NaN,WTIV,Fasten Tower Section,4.000000,8.333333e+04,ACTION,1305.523656,TurbineInstallation,NaN,TurbineInstallation,34.0,NaN,NaN,NaN,150.0,NaN
585,NaN,WTIV,Fasten Nacelle,4.000000,8.333333e+04,ACTION,1309.523656,TurbineInstallation,NaN,TurbineInstallation,34.0,NaN,NaN,NaN,150.0,NaN
588,NaN,WTIV,Fasten Blade,1.500000,3.125000e+04,ACTION,1311.023656,TurbineInstallation,NaN,TurbineInstallation,34.0,NaN,NaN,NaN,150.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2472,NaN,WTIV,Attach Blade,3.500000,7.291667e+04,ACTION,4544.996989,TurbineInstallation,NaN,TurbineInstallation,34.0,NaN,NaN,NaN,150.0,NaN
2473,NaN,WTIV,Release Blade,1.000000,2.083333e+04,ACTION,4545.996989,TurbineInstallation,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2474,NaN,WTIV,Lift Blade,1.500000,3.125000e+04,ACTION,4547.496989,TurbineInstallation,NaN,TurbineInstallation,34.0,NaN,NaN,NaN,150.0,NaN
2475,NaN,WTIV,Attach Blade,3.500000,7.291667e+04,ACTION,4550.996989,TurbineInstallation,NaN,TurbineInstallation,34.0,NaN,NaN,NaN,150.0,NaN


In [9]:
# Operations can also be grouped to see a total amount of time spend on each operation

turbine_install.groupby(["action"]).sum()['duration']

action
Attach Blade             420.000000
Attach Nacelle           240.000000
Attach Tower Section     480.000000
Fasten Blade             180.000000
Fasten Nacelle           160.000000
Fasten Tower Section     320.000000
Jackdown                  15.733333
Jackup                    15.733333
Lift Blade               180.000000
Lift Nacelle              60.000000
Lift Tower Section        90.000000
Mobilize                 168.000000
Position Onsite           80.000000
Reequip                   80.000000
Release Blade            120.000000
Release Nacelle          120.000000
Release Tower Section    240.000000
Transit                  452.400000
Name: duration, dtype: float64

In [10]:
turbine_install.groupby(["agent"]).sum()['duration']

agent
WTIV    3421.866667
Name: duration, dtype: float64

In [11]:
def installation_durations(df):
    import pandas as pd 
    install = df.phase.unique()
    df_final = pd.DataFrame()
    phase = list()
    duration = list()
    cost = list()
    agent = list()
    action = list()
    
    for i in install:
        df_phase = df[df["phase"] == i]
        agents = df_phase.agent.unique()
        
        #print (agents)
        phase.append(i)
        duration.append(df_phase["time"].max())
        cost.append(df_phase["time"].max()/(8760/12) * 2000000) 
        agent.append('Port')
        action.append('Port')
        for j in agents:
            df_agent = df_phase[df_phase["agent"] == j]
            actions = df_agent.action.unique()
            for k in actions:
                df_action = df_agent[df_agent["action"] == k]
                phase.append(i)
                duration.append(df_action["duration"].sum())
                cost.append(df_action["cost"].sum())
                agent.append(j)
                action.append(k)

    
    df_final = pd.DataFrame(list(zip(phase, agent, action, duration, cost)),
                   columns =['phase', 'agent', 'action', 'duration', 'cost'])

    df_final["day_rate"] = df_final["cost"] / (df_final["duration"]/(24))

    return df_final

In [12]:
df_1 = installation_durations(turbine_install)
display(df_1)
print(df_1["cost"].sum())
df_1.to_csv("data/turbine_output_15_MW.csv")

,phase,agent,action,duration,cost,day_rate
0,TurbineInstallation,Port,Port,4551.390323,1.246956e+07,65753.424658
1,TurbineInstallation,WTIV,Mobilize,168.000000,3.500000e+06,500000.000000
2,TurbineInstallation,WTIV,Fasten Tower Section,320.000000,6.666667e+06,500000.000000
3,TurbineInstallation,WTIV,Fasten Nacelle,160.000000,3.333333e+06,500000.000000
4,TurbineInstallation,WTIV,Fasten Blade,180.000000,3.750000e+06,500000.000000
5,TurbineInstallation,WTIV,Transit,452.400000,9.425000e+06,500000.000000
6,TurbineInstallation,WTIV,Position Onsite,80.000000,1.666667e+06,500000.000000
7,TurbineInstallation,WTIV,Jackup,15.733333,3.277778e+05,500000.000000
8,TurbineInstallation,WTIV,Release Tower Section,240.000000,5.000000e+06,500000.000000
9,TurbineInstallation,WTIV,Lift Tower Section,90.000000,1.875000e+06,500000.000000


83758451.41712795


In [13]:
monopile_install = df.loc[df['phase']=="MonopileInstallation"]
monopile_install.groupby(["action"]).sum()['duration']

action
Bolt TP                     160.000000
Crane Reequip                80.000000
Drive Monopile               60.000000
Fasten Monopile             480.000000
Fasten Transition Piece     320.000000
Jackdown                     15.733333
Jackup                       15.733333
Lower Monopile                0.176000
Lower TP                     40.000000
Mobilize                    168.000000
Position Onsite              80.000000
Release Monopile            120.000000
Release Transition Piece     80.000000
RovSurvey                    40.000000
Transit                     916.400000
Upend Monopile               33.860561
Name: duration, dtype: float64

In [14]:
df_2 = installation_durations(monopile_install)
display(df_2)
print(df_2["cost"].sum())
df_2.to_csv("data/monopile_output_15_MW.csv")

,phase,agent,action,duration,cost,day_rate
0,MonopileInstallation,Port,Port,3495.236561,9.575991e+06,65753.424658
1,MonopileInstallation,WTIV,Mobilize,168.000000,3.500000e+06,500000.000000
2,MonopileInstallation,WTIV,Fasten Monopile,480.000000,1.000000e+07,500000.000000
3,MonopileInstallation,WTIV,Fasten Transition Piece,320.000000,6.666667e+06,500000.000000
4,MonopileInstallation,WTIV,Transit,916.400000,1.909167e+07,500000.000000
5,MonopileInstallation,WTIV,Position Onsite,80.000000,1.666667e+06,500000.000000
6,MonopileInstallation,WTIV,Jackup,15.733333,3.277778e+05,500000.000000
7,MonopileInstallation,WTIV,RovSurvey,40.000000,8.333333e+05,500000.000000
8,MonopileInstallation,WTIV,Release Monopile,120.000000,2.500000e+06,500000.000000
9,MonopileInstallation,WTIV,Upend Monopile,33.860561,7.054284e+05,500000.000000


63948974.49737363
